# Strom: snímky → mračno bodů (.las) na GPU, v metrech

Rekonstrukce hustého mračna z dronových snímků pomocí **COLMAP (CUDA)** na Colab GPU,
se zarovnáním na GPS z `geo.txt` (`model_aligner`) → výstup je **přibližně v metrech**.

## Postup
1. Na Google Disku měj složku, např. `MyDrive/lidar/strom/`, a v ní `images/` (95 jpg + `geo.txt`).
2. **Runtime → Change runtime type → Hardware accelerator: GPU**.
3. Spusť buňky shora dolů (Runtime → Run all).
4. Výsledek `strom.las` se uloží zpět do `MyDrive/lidar/strom/`.

> ⚠️ **Měřítko:** GPS dronu kolísá za celý oblet jen ~1–2 m (≈ šum GPS), takže metry jsou
> jen **orientační** (klidně ±desítky %). Tvar je správný, absolutní rozměr ber s rezervou.

> ℹ️ COLMAP z apt je GUI build → i CLI příkazy potřebují `QT_QPA_PLATFORM=offscreen`,
> jinak spadnou na `could not connect to display`. Notebook to nastavuje za tebe.

## 1) Kontrola GPU
Musí vypsat NVIDIA (např. Tesla T4). Pokud ne → Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi

## 2) Instalace COLMAP (CUDA) + nástrojů

In [ ]:
!apt-get -qq update
!apt-get -qq install -y colmap
!pip -q install "laspy[laszip]" plyfile
!colmap -h | head -1

## 3) Připojení Disku + kopírování snímků
Uprav `DRIVE_DIR`, pokud máš složku jinak než `MyDrive/lidar/strom`.
Tady se taky nastaví `QT_QPA_PLATFORM=offscreen` (kvůli GUI buildu COLMAPu).

In [ ]:
from google.colab import drive
import os, shutil, glob
os.environ['QT_QPA_PLATFORM'] = 'offscreen'   # COLMAP CLI bez displeje
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/lidar/strom'   # <-- tvoje složka na Disku
SRC = os.path.join(DRIVE_DIR, 'images')
WORK = '/content/work'
IMAGES = os.path.join(WORK, 'images')
os.makedirs(WORK, exist_ok=True)
if os.path.exists(IMAGES):
    shutil.rmtree(IMAGES)
shutil.copytree(SRC, IMAGES)
n = len(glob.glob(os.path.join(IMAGES, '*.jpg')))
print('Snímků:', n)
assert n > 0, 'V images/ nejsou žádné jpg – zkontroluj DRIVE_DIR.'

## 4) GPS reference pro COLMAP
`geo.txt` (ODM) má formát `jméno lon lat alt`. COLMAP `model_aligner` chce `jméno lat lon alt` →
prohodíme sloupce do `geo_colmap.txt`.

In [ ]:
geo_in = os.path.join(IMAGES, 'geo.txt')
geo_out = os.path.join(WORK, 'geo_colmap.txt')
lines = []
for ln in open(geo_in):
    p = ln.split()
    if len(p) >= 4 and p[0].lower().endswith('.jpg'):
        name, lon, lat, alt = p[0], p[1], p[2], p[3]
        lines.append(f'{name} {lat} {lon} {alt}')   # COLMAP: lat lon alt
open(geo_out, 'w').write('\n'.join(lines) + '\n')
print('GPS referencí:', len(lines))
print('\n'.join(lines[:3]))

## 5) Řídká rekonstrukce (SfM): features → matching → mapper
`Q=QT_QPA_PLATFORM=offscreen` je před každým příkazem (jinak COLMAP spadne na displej).

In [ ]:
Q = 'QT_QPA_PLATFORM=offscreen'
DB = '/content/work/database.db'
!rm -f $DB && rm -rf /content/work/sparse /content/work/sparse_aligned /content/work/dense
!mkdir -p /content/work/sparse

!$Q colmap feature_extractor \
  --database_path $DB \
  --image_path /content/work/images \
  --ImageReader.single_camera 1 \
  --ImageReader.camera_model OPENCV \
  --SiftExtraction.use_gpu 1

!$Q colmap exhaustive_matcher --database_path $DB --SiftMatching.use_gpu 1

!$Q colmap mapper \
  --database_path $DB \
  --image_path /content/work/images \
  --output_path /content/work/sparse
!ls /content/work/sparse

## 6) Zarovnání na GPS → metry (`model_aligner`)
`enu` = lokální metrické souřadnice (East-North-Up). Když je málo inlierů (úzký oblet),
zkus zvýšit `alignment_max_error`. Pokud zarovnání selže, použij nezarovnaný `sparse/0`.

In [ ]:
!mkdir -p /content/work/sparse_aligned
ret = os.system(
  'QT_QPA_PLATFORM=offscreen colmap model_aligner '
  '--input_path /content/work/sparse/0 '
  '--output_path /content/work/sparse_aligned '
  '--ref_images_path /content/work/geo_colmap.txt '
  '--ref_is_gps 1 '
  '--alignment_type enu '
  '--alignment_max_error 5.0'
)
if ret == 0 and os.path.exists('/content/work/sparse_aligned/images.bin'):
    ALIGNED = '/content/work/sparse_aligned'
    print('Zarovnáno na GPS → metry.')
else:
    ALIGNED = '/content/work/sparse/0'
    print('⚠️ Zarovnání selhalo – pokračuju bez měřítka (libovolné jednotky).')
print('Použiju:', ALIGNED)

## 7) Husté mračno (MVS na GPU)
`max_image_size 1600` drží paměť v mezích T4 (16 GB). Na A100 (40 GB) klidně dej `2560`.
Tohle je nejdelší krok.

In [ ]:
!$Q colmap image_undistorter \
  --image_path /content/work/images \
  --input_path $ALIGNED \
  --output_path /content/work/dense \
  --output_type COLMAP \
  --max_image_size 1600

!$Q colmap patch_match_stereo \
  --workspace_path /content/work/dense \
  --workspace_format COLMAP \
  --PatchMatchStereo.geom_consistency true

!$Q colmap stereo_fusion \
  --workspace_path /content/work/dense \
  --workspace_format COLMAP \
  --input_type geometric \
  --output_path /content/work/dense/fused.ply
!ls -lh /content/work/dense/fused.ply

## 8) Převod PLY → LAS a uložení na Disk

In [ ]:
import numpy as np, laspy
from plyfile import PlyData

v = PlyData.read('/content/work/dense/fused.ply')['vertex']
xyz = np.vstack([v['x'], v['y'], v['z']]).T.astype('float64')
h = laspy.LasHeader(point_format=3, version='1.2')
h.offsets = xyz.min(axis=0)
h.scales = [0.001, 0.001, 0.001]
las = laspy.LasData(h)
las.x, las.y, las.z = xyz[:,0], xyz[:,1], xyz[:,2]
names = v.data.dtype.names
if 'red' in names:
    las.red   = v['red'].astype('uint16')   * 257
    las.green = v['green'].astype('uint16') * 257
    las.blue  = v['blue'].astype('uint16')  * 257
out = os.path.join(DRIVE_DIR, 'strom.las')
las.write(out)
print('Hotovo:', out, '| bodů:', len(xyz))
print('Rozsah (m):', np.round(xyz.max(0) - xyz.min(0), 2))

## 9) Stažení
`strom.las` je v `MyDrive/lidar/strom/` – stažni z Disku, nebo přímo z prohlížeče níže.

In [ ]:
from google.colab import files
files.download(os.path.join(DRIVE_DIR, 'strom.las'))